# Docflow — dots.mocr inference server (Colab dev tool)

Runs `dots.ocr` on Colab's GPU and exposes a `/parse` endpoint that the local Docflow
engine calls via `RemoteDotsModel`. This is a **throwaway development tool** for Phase 0 —
production inference becomes a Modal function in Phase 1.

**Usage**
1. Runtime → Change runtime type → **GPU** (T4 is enough).
2. Run every cell top to bottom.
3. The last cell prints a public URL. On your machine set it and run the CLI:
   - PowerShell: `$env:DOCFLOW_DOTS_URL="<url>"`
   - then: `uv run docflow convert file.pdf --model dots -o out/`

The response schema matches Docflow's `PageLayout` contract exactly:
`{page_index, image_width, image_height, elements: [{category, bbox, text}]}`.

In [ ]:
# 1. Dependencies
!pip install -q -U transformers accelerate qwen_vl_utils huggingface_hub flask flask-cors pyngrok pillow

In [ ]:
# 2. Download weights to a dot-free folder.
# The HF repo id 'rednote-hilab/dots.ocr' contains a '.', which breaks transformers'
# trust_remote_code dynamic import. Downloading to ./DotsOCR sidesteps that.
from huggingface_hub import snapshot_download

MODEL_DIR = snapshot_download(repo_id="rednote-hilab/dots.ocr", local_dir="./DotsOCR")
print("weights at:", MODEL_DIR)

In [ ]:
# 3. Load the model and processor.
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

model = AutoModelForCausalLM.from_pretrained(
    "./DotsOCR",
    attn_implementation="sdpa",  # avoids a flash-attn build on Colab
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained("./DotsOCR", trust_remote_code=True)
print("model loaded")

In [ ]:
# 4. Inference: one PIL image -> list of layout elements.
import json
import re
from qwen_vl_utils import process_vision_info

# Faithful rendering of dots.ocr's prompt_layout_all_en. If you have the official
# dots_ocr package, you can replace this with its exact prompt string.
PROMPT = (
    "Please output the layout information from this image, including each layout "
    "element's bbox, its category, and the corresponding text content.\n\n"
    "1. Bbox format: [x1, y1, x2, y2] in pixels of this image.\n"
    "2. Categories: ['Caption','Footnote','Formula','List-item','Page-footer',"
    "'Page-header','Picture','Section-header','Table','Text','Title'].\n"
    "3. Text rules: Picture -> omit text; Formula -> LaTeX; Table -> HTML; others -> Markdown.\n"
    "4. Output a single JSON array of elements, sorted in human reading order."
)


def _extract_json_array(text: str):
    """Pull the JSON array out of the model's decoded text (tolerates code fences)."""
    match = re.search(r"\[.*\]", text, re.DOTALL)
    return json.loads(match.group(0)) if match else []


def parse_image(image):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    generated = model.generate(**inputs, max_new_tokens=24000)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    decoded = processor.batch_decode(trimmed, skip_special_tokens=True)[0]
    return _extract_json_array(decoded)

In [ ]:
# 5. Flask app exposing /parse and /health.
import io
import threading
from flask import Flask, jsonify, request
from flask_cors import CORS
from PIL import Image

app = Flask(__name__)
CORS(app)


@app.get("/health")
def health():
    return jsonify(status="ok")


@app.post("/parse")
def parse():
    page_index = int(request.form.get("page_index", 0))
    image = Image.open(io.BytesIO(request.files["image"].read())).convert("RGB")
    elements = parse_image(image)
    return jsonify(
        page_index=page_index,
        image_width=image.width,
        image_height=image.height,
        elements=[
            {"category": e["category"], "bbox": e["bbox"], "text": e.get("text")}
            for e in elements
        ],
    )


def _serve():
    app.run(host="0.0.0.0", port=5000, threaded=True)


threading.Thread(target=_serve, daemon=True).start()
print("Flask serving on :5000")

## 6. Expose a public URL

Uses **pyngrok**. Get a free authtoken at <https://dashboard.ngrok.com> and paste it below.
(Prefer no signup? Replace this cell with a `cloudflared` quick tunnel instead.)

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("PASTE_YOUR_NGROK_AUTHTOKEN_HERE")
public_url = ngrok.connect(5000).public_url
print("DOCFLOW_DOTS_URL =", public_url)
print("\nOn your machine (PowerShell):")
print(f'  $env:DOCFLOW_DOTS_URL="{public_url}"')
print("  uv run docflow convert file.pdf --model dots -o out/")